In [1]:
"""Unsupervised interval features.

A transformer for extracting all interval features produced by an
R-STSF-style recursive splitting process, but without supervised pruning.
"""

__maintainer__ = []
__all__ = ["UnsupervisedIntervals"]

import inspect

import numpy as np
from joblib import Parallel, delayed
from sklearn.utils import check_random_state

from aeon.base._base import _clone_estimator
from aeon.transformations.base import BaseTransformer
from aeon.transformations.collection.base import BaseCollectionTransformer
from aeon.utils.numba.general import z_normalise_series_3d
from aeon.utils.numba.stats import (
    row_count_above_mean,
    row_count_mean_crossing,
    row_iqr,
    row_mean,
    row_median,
    row_numba_max,
    row_numba_min,
    row_slope,
    row_std,
)
from aeon.utils.validation import check_n_jobs


class UnsupervisedIntervals(BaseCollectionTransformer):
    """Unsupervised interval feature transformer.

    Extracts interval features using the same recursive splitting idea as
    SupervisedIntervals, but instead of keeping only the better-scoring half,
    it keeps both halves and continues recursively on both branches.

    This produces the full interval tree induced by the splitting process for
    each channel, feature, and interval-generation run.

    Parameters
    ----------
    n_intervals : int, default=50
        Number of interval-generation runs. Each run starts from a fresh random
        initial cut point and recursively expands both sides.
    min_interval_length : int, default=3
        Minimum length of extracted intervals. Minimum effective value is 3.
    features : callable, list of callables, default=None
        Functions used to extract features from intervals. Must take a 2D array
        of shape (n_cases, interval_length) and return a 1D array of shape
        (n_cases,). If None, defaults to:
        [mean, median, std, slope, min, max, iqr,
         count_mean_crossing, count_above_mean]
    randomised_split_point : bool, default=True
        If True, recursive splits are randomised subject to min_interval_length.
        If False, intervals are split in half.
    normalise_for_search : bool, default=True
        If True, splitting recursion is based on normalised data, but output
        features are computed on the original data.
    random_state : None, int or RandomState, default=None
        Random seed.
    n_jobs : int, default=1
        Number of jobs to use for fit and transform.
    parallel_backend : str, ParallelBackendBase instance or None, default=None
        joblib backend.

    Attributes
    ----------
    n_cases_ : int
        Number of cases in training data.
    n_channels_ : int
        Number of channels.
    n_timepoints_ : int
        Series length.
    intervals_ : list of tuples
        Each tuple is (start, end, dim, feature). One tuple per output column.
    """

    _tags = {
        "output_data_type": "Tabular",
        "capability:multivariate": True,
        "capability:multithreading": True,
        "requires_y": False,
        "algorithm_type": "interval",
    }

    # if features contains a transformer, it must contain a parameter name from
    # transformer_feature_selection and an attribute name (or property) from
    # transformer_feature_names to allow a single feature to be transformed at a time.
    transformer_feature_selection = ["features"]
    transformer_feature_names = [
        "features_arguments_",
        "_features_arguments",
        "get_features_arguments",
        "_get_features_arguments",
    ]

    def __init__(
        self,
        n_intervals=50,
        min_interval_length=3,
        features=None,
        randomised_split_point=True,
        normalise_for_search=True,
        random_state=None,
        n_jobs=1,
        parallel_backend=None,
    ):
        self.n_intervals = n_intervals
        self.min_interval_length = min_interval_length
        self.features = features
        self.randomised_split_point = randomised_split_point
        self.normalise_for_search = normalise_for_search
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.parallel_backend = parallel_backend

        super().__init__()

    def _fit_transform(self, X, y=None):
        X, rng = self._fit_setup(X)

        X_norm = z_normalise_series_3d(X) if self.normalise_for_search else X

        fit = Parallel(
            n_jobs=self._n_jobs, backend=self.parallel_backend, prefer="threads"
        )(
            delayed(self._generate_intervals)(
                X,
                X_norm,
                rng.randint(np.iinfo(np.int32).max),
                True,
            )
            for _ in range(self.n_intervals)
        )

        intervals, transformed_intervals = zip(*fit)

        for ints in intervals:
            self.intervals_.extend(ints)

        self._transform_features = [True] * len(self.intervals_)

        Xt = transformed_intervals[0]
        for i in range(1, self.n_intervals):
            Xt = np.hstack((Xt, transformed_intervals[i]))

        return Xt

    def _fit(self, X, y=None):
        X, rng = self._fit_setup(X)

        X_norm = z_normalise_series_3d(X) if self.normalise_for_search else X

        fit = Parallel(
            n_jobs=self._n_jobs, backend=self.parallel_backend, prefer="threads"
        )(
            delayed(self._generate_intervals)(
                X,
                X_norm,
                rng.randint(np.iinfo(np.int32).max),
                False,
            )
            for _ in range(self.n_intervals)
        )

        intervals, _ = zip(*fit)

        for ints in intervals:
            self.intervals_.extend(ints)

        self._transform_features = [True] * len(self.intervals_)

        return self

    def _transform(self, X, y=None):
        transform = Parallel(
            n_jobs=self._n_jobs, backend=self.parallel_backend, prefer="threads"
        )(
            delayed(self._transform_intervals)(X, i)
            for i in range(len(self.intervals_))
        )

        Xt = np.zeros((X.shape[0], len(transform)))
        for i, t in enumerate(transform):
            Xt[:, i] = t

        return Xt

    def _fit_setup(self, X):
        self.intervals_ = []

        self.n_cases_, self.n_channels_, self.n_timepoints_ = X.shape
        self._n_jobs = check_n_jobs(self.n_jobs)

        if self.n_cases_ <= 0:
            raise ValueError("UnsupervisedIntervals requires at least 1 time series.")

        self._min_interval_length = max(3, self.min_interval_length)

        if self._min_interval_length * 2 > self.n_timepoints_:
            raise ValueError(
                "Minimum interval length must allow at least one valid split."
            )

        self._features = self.features
        if self.features is None:
            self._features = [
                row_mean,
                row_median,
                row_std,
                row_slope,
                row_numba_min,
                row_numba_max,
                row_iqr,
                row_count_mean_crossing,
                row_count_above_mean,
            ]

        if not isinstance(self._features, list):
            self._features = [self._features]

        rng = check_random_state(self.random_state)

        msg = (
            "Transformers must have a parameter from 'transformer_feature_selection' "
            "to allow selecting single features, and a list of feature names in "
            "'transformer_feature_names'. Transformers which require 'fit' are "
            "currently unsupported."
        )

        expanded_features = []
        for f in self._features:
            if callable(f):
                expanded_features.append(f)
            elif isinstance(f, BaseTransformer):
                if not f.get_tag("fit_is_empty"):
                    raise ValueError(msg)

                params = inspect.signature(f.__init__).parameters

                att_name = None
                for n in self.transformer_feature_selection:
                    if params.get(n, None) is not None:
                        att_name = n
                        break

                if att_name is None:
                    raise ValueError(msg)

                t_features = None
                for n in self.transformer_feature_names:
                    if hasattr(f, n) and isinstance(getattr(f, n), (list, tuple)):
                        t_features = getattr(f, n)
                        break

                if t_features is None:
                    raise ValueError(msg)

                for t_f in t_features:
                    new_transformer = _clone_estimator(f, rng)
                    setattr(new_transformer, att_name, t_f)
                    expanded_features.append(new_transformer)
            else:
                raise ValueError("Features must be callables or BaseTransformer instances.")

        self._features = expanded_features
        return X, rng

    def _generate_intervals(self, X, X_norm, seed, keep_transform):
        rng = check_random_state(seed)

        Xt = np.empty((self.n_cases_, 0)) if keep_transform else None
        intervals = []

        for dim in range(self.n_channels_):
            for feature in self._features:
                random_cut_point = int(rng.randint(1, self.n_timepoints_ - 1))
                while (
                    random_cut_point < self._min_interval_length
                    or self.n_timepoints_ - random_cut_point < self._min_interval_length
                ):
                    random_cut_point = int(rng.randint(1, self.n_timepoints_ - 1))

                intervals_L, Xt_L = self._unsupervised_search(
                    X_norm[:, dim, :random_cut_point],
                    0,
                    feature,
                    dim,
                    X[:, dim, :],
                    rng,
                    keep_transform,
                    isinstance(feature, BaseTransformer),
                )
                intervals.extend(intervals_L)

                if keep_transform and Xt_L.shape[1] > 0:
                    Xt = np.hstack((Xt, Xt_L))

                intervals_R, Xt_R = self._unsupervised_search(
                    X_norm[:, dim, random_cut_point:],
                    random_cut_point,
                    feature,
                    dim,
                    X[:, dim, :],
                    rng,
                    keep_transform,
                    isinstance(feature, BaseTransformer),
                )
                intervals.extend(intervals_R)

                if keep_transform and Xt_R.shape[1] > 0:
                    Xt = np.hstack((Xt, Xt_R))

        return intervals, Xt

    def _transform_intervals(self, X, idx):
        if not self._transform_features[idx]:
            return np.zeros(X.shape[0])

        start, end, dim, feature = self.intervals_[idx]

        if isinstance(feature, BaseTransformer):
            return feature.transform(X[:, dim, start:end]).flatten()
        else:
            return feature(X[:, dim, start:end])

    def _compute_feature(self, feature, X_interval, feature_is_transformer):
        if feature_is_transformer:
            return feature.transform(X_interval).flatten()
        return feature(X_interval)

    def _unsupervised_search(
        self,
        X,
        ini_idx,
        feature,
        dim,
        X_ori,
        rng,
        keep_transform,
        feature_is_transformer,
    ):
        intervals = []
        Xt = np.empty((X.shape[0], 0)) if keep_transform else None

        # stop if interval too short to split into two valid children
        if X.shape[1] < self._min_interval_length * 2:
            return intervals, Xt

        if self.randomised_split_point and X.shape[1] != self._min_interval_length * 2:
            div_point = rng.randint(
                self._min_interval_length, X.shape[1] - self._min_interval_length
            )
        else:
            div_point = int(X.shape[1] / 2)

        sub_interval_0 = X[:, :div_point]
        sub_interval_1 = X[:, div_point:]

        start_0 = ini_idx
        end_0 = ini_idx + sub_interval_0.shape[1]

        start_1 = ini_idx + div_point
        end_1 = start_1 + sub_interval_1.shape[1]

        # keep both children
        intervals.append((start_0, end_0, dim, feature))
        intervals.append((start_1, end_1, dim, feature))

        if keep_transform:
            feat_0 = self._compute_feature(
                feature, X_ori[:, start_0:end_0], feature_is_transformer
            )
            feat_1 = self._compute_feature(
                feature, X_ori[:, start_1:end_1], feature_is_transformer
            )

            Xt = np.hstack(
                (
                    Xt,
                    feat_0.reshape(feat_0.shape[0], 1),
                    feat_1.reshape(feat_1.shape[0], 1),
                )
            )

        # recurse on both branches
        intervals_L, Xt_L = self._unsupervised_search(
            sub_interval_0,
            start_0,
            feature,
            dim,
            X_ori,
            rng,
            keep_transform,
            feature_is_transformer,
        )
        intervals.extend(intervals_L)

        if keep_transform and Xt_L.shape[1] > 0:
            Xt = np.hstack((Xt, Xt_L))

        intervals_R, Xt_R = self._unsupervised_search(
            sub_interval_1,
            start_1,
            feature,
            dim,
            X_ori,
            rng,
            keep_transform,
            feature_is_transformer,
        )
        intervals.extend(intervals_R)

        if keep_transform and Xt_R.shape[1] > 0:
            Xt = np.hstack((Xt, Xt_R))

        return intervals, Xt

    def set_features_to_transform(self, arr, raise_error=True):
        """Set transform_features to the given array.

        Each index in the list corresponds to the index of an interval.
        True intervals are included in transform, False intervals are skipped
        and their output is set to 0.

        Parameters
        ----------
        arr : list of bool
            A list indicating which intervals to keep.
        raise_error : bool, default=True
            Whether to raise an error or return False if input is invalid.

        Returns
        -------
        completed : bool
            Whether the operation was successful.
        """
        if len(arr) != len(self.intervals_) or not all(isinstance(b, bool) for b in arr):
            if raise_error:
                raise ValueError(
                    "Input must be a list of bools of length len(intervals_)."
                )
            return False

        self._transform_features = arr
        return True

    @classmethod
    def _get_test_params(cls, parameter_set="default"):
        """Return testing parameter settings for the estimator."""
        if parameter_set == "results_comparison":
            return {
                "n_intervals": 1,
                "randomised_split_point": True,
            }
        return {
            "n_intervals": 1,
            "randomised_split_point": False,
        }

In [2]:
from aeon.transformations.collection.interval_based import SupervisedIntervals, RandomIntervals

t = UnsupervisedIntervals(n_jobs=-1)
s = SupervisedIntervals(n_jobs=-1)
r = RandomIntervals(n_intervals=600, n_jobs=-1)

In [3]:
from tscglue import data_loader


dataset = "ToeSegmentation1"
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 7)

In [4]:
#print("UnsupervisedIntervals:", t.fit_transform(X_train).shape)

In [ ]:
from sklearn.random_projection import GaussianRandomProjection
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

X_unsup_train = t.fit_transform(X_train)
X_unsup_test = t.transform(X_test)
rp = GaussianRandomProjection(random_state=42)
rp.fit(X_unsup_train)
X_unsup_rp_train = rp.transform(X_unsup_train)
X_unsup_rp_test = rp.transform(X_unsup_test)

X_sup_train = s.fit_transform(X_train, y_train)
X_sup_test = s.transform(X_test)

X_rand_train = r.fit_transform(X_train)
X_rand_test = r.transform(X_test)

# Concatenate UnsupIntervals + GaussianRP with RandomIntervals
X_combined_train = np.hstack((X_unsup_rp_train, X_rand_train))
X_combined_test = np.hstack((X_unsup_rp_test, X_rand_test))

print(f"UnsupIntervals + RP: {X_unsup_rp_train.shape[1]} features")
print(f"SupervisedIntervals: {X_sup_train.shape[1]} features")
print(f"RandomIntervals:     {X_rand_train.shape[1]} features")
print(f"UnsupRP + Random:    {X_combined_train.shape[1]} features")

datasets = [
    ("UnsupIntervals + GaussianRP", X_unsup_rp_train, X_unsup_rp_test),
    ("SupervisedIntervals",         X_sup_train,      X_sup_test),
    ("RandomIntervals",             X_rand_train,     X_rand_test),
    ("UnsupRP + RandomIntervals",   X_combined_train, X_combined_test),
]

classifiers = [
    ("ExtraTrees",         lambda: ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42)),
    ("ExtraTrees+PCA(95%)", lambda: make_pipeline(PCA(n_components=0.95), ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42))),
    ("RidgeCV",            lambda: RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
]

for clf_name, clf_fn in classifiers:
    print(f"\n--- {clf_name} ---")
    for name, Xtr, Xte in datasets:
        clf = clf_fn()
        clf.fit(Xtr, y_train)
        acc = accuracy_score(y_test, clf.predict(Xte))
        print(f"{name:30s} -> accuracy: {acc:.4f}")

In [ ]:
#print("RandomIntervals:", r.fit_transform(X_train).shape)